# Read Library

In [2]:
%load_ext autoreload
%autoreload 2

In [5]:
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.preprocessing import StandardScaler

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torchinfo import summary
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math
from scipy import stats
import warnings
import re
import datetime
from package.utils import profile_data ,DotDict, timer, get_config, load_data, merge_data ,save_file

In [8]:
from package.model_deep_ae_20250911 import (set_seed,
                                generate_mock_data,
                                expand_rule_columns,
                                subset_by_timeunit,
                                split_train_test,
                                prepare_train_val_test_data,
                                Autoencoder,
                                save_checkpoint,
                                load_checkpoint,
                                train_autoencoder_checkpoint,
                                load_pickled_autoencoder,
                                plot_learning_curve,
                                get_reconstruction_error,
                                flag_anomalies,
                                extract_embeddings,
                                plot_latent_space_2d,
                                plot_latent_space_3d,
                                create_pack_results,
                                create_pack_results_date,
                                evaluate_fraud_predictions,
                                evaluate_thresholds,
                                get_thresholds,
                                map_errors_to_percentiles
                                )

# Read Data

In [9]:
mock1 = generate_mock_data(n_sales=10000, n_rules=10, fraud_ratio=0.05, seed=42)
mock1

function: generate_mock_data is starting...
function: generate_mock_data successfully executed at 1.6293511390686035s


,sales_id,expected_dt,timeunit,flag_fraud,rule_1,rule_2,rule_3,rule_4,rule_5,rule_6,rule_7,rule_8,rule_9,rule_10
0,0,2024-12-30,l3,0,0.360016,-0.228151,0.282450,-0.724385,-0.585063,1.173842,-1.051548,-0.977754,-0.051581,-0.771890
1,1,2024-07-14,l3,1,2.272585,-0.017009,5.743308,2.880759,5.058952,4.337591,-0.622909,3.658322,2.477348,1.960721
2,2,2025-05-26,l3,0,1.024774,0.908937,-0.997060,-1.057251,-0.411173,-0.026746,-0.657898,1.040642,0.347390,0.406085
3,3,2025-05-25,l3,0,-1.266539,1.167291,-2.820169,0.834055,0.934312,-0.432401,1.272910,-0.063098,0.062432,0.339529
4,4,2025-06-17,l3,0,3.186575,1.344434,0.701131,-0.912804,0.143590,-0.813124,0.822111,0.140467,-1.124522,0.081125
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59995,9995,2024-07-01,l36,0,-1.393785,1.119496,-2.225198,0.355354,-1.114918,-0.348127,0.696269,-1.856022,-0.773980,-1.019967
59996,9996,2025-05-03,l36,0,1.914040,-0.250864,-0.670143,0.249901,-0.351280,-0.121383,0.274123,-0.559135,-0.084692,0.955931
59997,9997,2024-10-22,l36,0,0.580123,-0.555800,0.536215,-0.590000,-1.563657,0.052191,0.271158,-0.696574,-1.674949,-1.058655
59998,9998,2024-11-24,l36,0,1.381821,-0.886731,-0.431143,0.390082,0.668774,-1.953408,-0.687125,-1.467512,-0.240759,0.654136


### Prep data

In [10]:
df_transformed = expand_rule_columns(mock1)
df_transformed

function: expand_rule_columns is starting...
function: expand_rule_columns successfully executed at 0.633152961730957s


,sales_id,expected_dt,flag_fraud,rule_10_l12,rule_10_l24,rule_10_l3,rule_10_l36,rule_10_l6,rule_10_l9,rule_1_l12,...,rule_8_l3,rule_8_l36,rule_8_l6,rule_8_l9,rule_9_l12,rule_9_l24,rule_9_l3,rule_9_l36,rule_9_l6,rule_9_l9
0,0,2024-12-30,0,1.294071,-0.033099,-0.771890,1.027978,-1.716319,-0.438359,-0.844588,...,-0.977754,-1.817574,-1.378777,-0.720249,-0.153002,0.147629,-0.051581,-1.928613,-1.005889,-1.700068
1,1,2024-07-14,1,2.970241,1.546394,1.960721,0.327022,1.395264,2.302689,2.134491,...,3.658322,2.169982,4.625339,4.100241,5.391337,4.974683,2.477348,5.201170,0.888200,0.345258
2,2,2025-05-26,0,-0.962456,0.007673,0.406085,-0.324495,-0.351193,-0.073130,-1.225513,...,1.040642,-0.017216,3.053033,1.115441,-2.003674,-0.515232,0.347390,1.687974,-0.886053,0.373724
3,3,2025-05-25,0,-0.557176,0.152205,0.339529,0.084018,0.929277,0.426988,0.121433,...,-0.063098,1.076034,-0.388208,1.075950,-0.600473,-2.693087,0.062432,-0.566217,0.786089,1.194925
4,4,2025-06-17,0,-0.822350,-0.840434,0.081125,-0.962901,-0.471622,-0.433285,1.266659,...,0.140467,-1.761751,-0.281367,1.718366,0.989666,-0.659094,-1.124522,-0.808209,-1.347342,0.004178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9995,2024-07-01,0,1.743071,-0.215017,-0.953072,-1.019967,-0.682181,1.137971,-0.507696,...,-0.945883,-1.856022,-1.172909,0.589566,0.158376,0.376485,-1.294563,-0.773980,0.137205,-0.786161
9996,9996,2025-05-03,0,0.185918,0.366925,-0.035507,0.955931,-0.901071,-0.267223,-1.110620,...,0.152714,-0.559135,-0.246215,0.267680,-0.556548,1.367107,0.888219,-0.084692,-0.196889,-2.770211
9997,9997,2024-10-22,0,-1.890159,-0.300070,0.348756,-1.058655,1.505729,0.651137,1.225570,...,0.762028,-0.696574,1.745624,-2.129891,0.216331,-1.248463,1.035880,-1.674949,0.124384,-0.070152
9998,9998,2024-11-24,0,-0.321089,0.573540,-0.461097,0.654136,0.506189,-1.361413,0.461321,...,-1.386205,-1.467512,-1.396175,0.997792,0.060714,0.336269,0.143255,-0.240759,-2.010408,-1.628413


In [14]:
x_df = df_transformed.drop(columns=['sales_id','expected_dt','flag_fraud'])
x_df

,rule_10_l12,rule_10_l24,rule_10_l3,rule_10_l36,rule_10_l6,rule_10_l9,rule_1_l12,rule_1_l24,rule_1_l3,rule_1_l36,...,rule_8_l3,rule_8_l36,rule_8_l6,rule_8_l9,rule_9_l12,rule_9_l24,rule_9_l3,rule_9_l36,rule_9_l6,rule_9_l9
0,1.294071,-0.033099,-0.771890,1.027978,-1.716319,-0.438359,-0.844588,-1.228917,0.360016,-0.485965,...,-0.977754,-1.817574,-1.378777,-0.720249,-0.153002,0.147629,-0.051581,-1.928613,-1.005889,-1.700068
1,2.970241,1.546394,1.960721,0.327022,1.395264,2.302689,2.134491,1.395910,2.272585,3.306197,...,3.658322,2.169982,4.625339,4.100241,5.391337,4.974683,2.477348,5.201170,0.888200,0.345258
2,-0.962456,0.007673,0.406085,-0.324495,-0.351193,-0.073130,-1.225513,-0.298250,1.024774,0.377113,...,1.040642,-0.017216,3.053033,1.115441,-2.003674,-0.515232,0.347390,1.687974,-0.886053,0.373724
3,-0.557176,0.152205,0.339529,0.084018,0.929277,0.426988,0.121433,2.608182,-1.266539,-0.638100,...,-0.063098,1.076034,-0.388208,1.075950,-0.600473,-2.693087,0.062432,-0.566217,0.786089,1.194925
4,-0.822350,-0.840434,0.081125,-0.962901,-0.471622,-0.433285,1.266659,1.266076,3.186575,-0.683313,...,0.140467,-1.761751,-0.281367,1.718366,0.989666,-0.659094,-1.124522,-0.808209,-1.347342,0.004178
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1.743071,-0.215017,-0.953072,-1.019967,-0.682181,1.137971,-0.507696,-1.086355,1.243766,-1.393785,...,-0.945883,-1.856022,-1.172909,0.589566,0.158376,0.376485,-1.294563,-0.773980,0.137205,-0.786161
9996,0.185918,0.366925,-0.035507,0.955931,-0.901071,-0.267223,-1.110620,-0.922528,-1.228504,1.914040,...,0.152714,-0.559135,-0.246215,0.267680,-0.556548,1.367107,0.888219,-0.084692,-0.196889,-2.770211
9997,-1.890159,-0.300070,0.348756,-1.058655,1.505729,0.651137,1.225570,-1.003623,-1.213162,0.580123,...,0.762028,-0.696574,1.745624,-2.129891,0.216331,-1.248463,1.035880,-1.674949,0.124384,-0.070152
9998,-0.321089,0.573540,-0.461097,0.654136,0.506189,-1.361413,0.461321,-1.034207,-0.321970,1.381821,...,-1.386205,-1.467512,-1.396175,0.997792,0.060714,0.336269,0.143255,-0.240759,-2.010408,-1.628413


In [15]:
scaler = StandardScaler()
scaler.fit(x_df)

,copy,True
,with_mean,True
,with_std,True


In [20]:
new = [
    x_df.iloc[1,:] # index 1, so score must be 0 in this record
]

In [21]:
new

[rule_10_l12    2.970241
 rule_10_l24    1.546394
 rule_10_l3     1.960721
 rule_10_l36    0.327022
 rule_10_l6     1.395264
 rule_10_l9     2.302689
 rule_1_l12     2.134491
 rule_1_l24     1.395910
 rule_1_l3      2.272585
 rule_1_l36     3.306197
 rule_1_l6      1.625940
 rule_1_l9      6.203698
 rule_2_l12     0.612803
 rule_2_l24     5.487387
 rule_2_l3     -0.017009
 rule_2_l36     3.344081
 rule_2_l6      3.666518
 rule_2_l9      3.731870
 rule_3_l12     3.207197
 rule_3_l24     1.084459
 rule_3_l3      5.743308
 rule_3_l36     1.605523
 rule_3_l6      3.859697
 rule_3_l9      1.455009
 rule_4_l12     5.603640
 rule_4_l24     3.339942
 rule_4_l3      2.880759
 rule_4_l36     1.222486
 rule_4_l6      4.176624
 rule_4_l9      2.570436
 rule_5_l12     4.737337
 rule_5_l24     3.479610
 rule_5_l3      5.058952
 rule_5_l36     1.317785
 rule_5_l6      5.030866
 rule_5_l9      2.063194
 rule_6_l12     5.412653
 rule_6_l24     4.032778
 rule_6_l3      4.337591
 rule_6_l36     2.734447


In [22]:
new = scaler.transform(new)
existing = scaler.transform(x_df)

C:\Users\alice.var\AppData\Local\miniconda3\envs\ad-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [23]:
new

array([[ 2.27214583,  1.15353147,  1.48223392,  0.16205582,  1.04320784,
         1.78491173,  1.6106441 ,  1.03840363,  1.71828389,  2.62804464,
         1.21266217,  4.81229343,  0.38297426,  4.36938564, -0.13050117,
         2.60220403,  2.86392261,  2.89726678,  2.49841922,  0.74641268,
         4.5332789 ,  1.17773875,  3.04333632,  1.07488804,  4.43326166,
         2.62248173,  2.27520461,  0.87549967,  3.28635169,  1.97981874,
         3.70583509,  2.67464584,  3.99383698,  0.94171963,  3.97189614,
         1.59551126,  4.27755585,  3.17361693,  3.41397869,  2.09238474,
         2.92718756,  1.00508839,  1.23178344,  2.09715253, -0.62075796,
         0.61577712,  0.83020059,  4.40234373,  0.66791039,  3.33092617,
         2.84852474,  1.63364965,  3.59622013,  3.20597028,  4.25949744,
         3.95832982,  1.87906838,  4.10748535,  0.60118494,  0.16323547]])

In [24]:
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.pdist.html#scipy.spatial.distance.pdist
metrics = ["euclidean", "cosine", "mahalanobis"]
idx = 0

In [25]:
# test output
sim = pairwise_distances(new, existing, metric=metrics[idx])
sim_df = pd.DataFrame(sim)
sim_df.T.round(4)

,0
0,24.0172
1,0.0000
2,22.2159
3,21.7248
4,22.6029
...,...
9995,22.9035
9996,22.4190
9997,22.9038
9998,23.5490


In [26]:
# Test output
sim = pairwise_distances(existing, new, metric=metrics[0])
sim_df = pd.DataFrame(sim, columns=["scores"])
sim_df.round(4).sort_values("scores")

,scores
1,0.0000
1727,11.6114
7598,11.9668
3091,12.1301
3402,12.2264
...,...
8316,24.6341
5886,24.7090
8567,24.8029
2085,24.9854


In [27]:
proxy = pd.DataFrame()
# Calculate the inverse covariance matrix for the mahalanobis distance
inv_cov_matrix = np.linalg.inv(np.cov(existing.T))

for metric in metrics:
    if metric == "mahalanobis":
        sim = pairwise_distances(existing, new, metric=metric, VI=inv_cov_matrix)
    else:
        sim = pairwise_distances(existing, new, metric=metric)
    proxy[metric] = sim.ravel()

In [29]:
result = proxy.round(4)
result

,euclidean,cosine,mahalanobis
0,24.0172,1.4344,15.4796
1,0.0000,0.0000,0.0000
2,22.2159,1.1371,15.6853
3,21.7248,1.0609,15.8976
4,22.6029,1.1473,14.9528
...,...,...,...
9995,22.9035,1.2442,15.4519
9996,22.4190,1.1347,14.3148
9997,22.9038,1.2538,15.0080
9998,23.5490,1.3930,14.8694


In [32]:
packed_df = pd.concat([df_transformed.loc[:,['sales_id','expected_dt','flag_fraud']],result],axis=1)
packed_df

,sales_id,expected_dt,flag_fraud,euclidean,cosine,mahalanobis
0,0,2024-12-30,0,24.0172,1.4344,15.4796
1,1,2024-07-14,1,0.0000,0.0000,0.0000
2,2,2025-05-26,0,22.2159,1.1371,15.6853
3,3,2025-05-25,0,21.7248,1.0609,15.8976
4,4,2025-06-17,0,22.6029,1.1473,14.9528
...,...,...,...,...,...,...
9995,9995,2024-07-01,0,22.9035,1.2442,15.4519
9996,9996,2025-05-03,0,22.4190,1.1347,14.3148
9997,9997,2024-10-22,0,22.9038,1.2538,15.0080
9998,9998,2024-11-24,0,23.5490,1.3930,14.8694
